# 06. Avaliação final (teste usado uma única vez)

Roteiro da última etapa do protocolo:

1. desenvolvimento concluído (`run_validation`, notebooks 02, 03 e 05);
2. protocolo congelado no terminal, na raiz do repositório:
   ```
   python -m src.pipeline.freeze
   ```
   que grava `results/protocolo.lock` com os hashes de todos os arquivos versionados em `src/**/*.py` e `data/*.py` e de `data/selecao_datasets.txt`, o espaço de busca, o orçamento e o commit;
3. **lock registrado no git ANTES da avaliação final**:
   ```
   git add results/protocolo.lock
   git commit -m "Congela o protocolo"
   ```
4. **avaliação final única** (`final_test`): cada sistema é ajustado no treino de cada fold externo e avaliado uma vez no teste;
5. **resultados registrados no git IMEDIATAMENTE**:
   ```
   git add results
   git commit -m "Avaliação final"
   ```
6. conferência do histórico: `python -m src.pipeline.check_protocol`.

`final_test` só executa em um repositório git, sem alterações não comitadas em `src/`, `data/` e no lock, com o lock versionado e idêntico ao de `HEAD`, com os hashes conferindo e sem nenhum vestígio de avaliação anterior (`test.csv`, `test_log.csv`, `test_manifest.json`, `preds/` ou commit com resultados). Use `--resume` somente para completar combinações pendentes ou com erro, com o mesmo lock. As travas locais são dissuasórias; a evidência do protocolo é o histórico git. Depois da avaliação final, não altere o modelo, o espaço de busca nem a seleção: o relatório deve apresentar o resultado tal como obtido.

In [1]:
import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

import json

import pandas as pd

from src.pipeline import check_protocol, final_test
from src.pipeline.freeze import LOCK_FILE, lock_digest, read_lock, verify_lock

RESULTS = REPO / 'results'

## 1. Conferir o lock

In [2]:
lock_path = RESULTS / LOCK_FILE
lock = read_lock(lock_path)
print('sha256 do lock:', lock_digest(lock_path))
print('commit:', lock['git_commit'], '| alterações não registradas:', lock['git_dirty'],
      '| arquivos travados:', len(lock['files']))
print('orçamento:', lock['budget'], '| semente:', lock['seed'])
print('sistemas:', lock['models'], '| folds externos:', lock['folds'], '| datasets:', len(lock['task_ids']))
problems = verify_lock(lock, REPO)
print('Lock íntegro.' if not problems else problems)

sha256 do lock: 3fb961581e2de87b260c1bb4b45ffca7dd91e36c4ae69e4a4473bdb367dae1e7
commit: f474ef331d4cd17df4f8b87acfea6fea1ebbac2d | alterações não registradas: False | arquivos travados: 30
orçamento: {'n_trials': 20, 'tune_timeout': 600, 'tune_max_samples': 10000} | semente: 42
sistemas: ['lightgbm', 'xgboost', 'catboost', 'group_default', 'group_tuned'] | folds externos: [0, 1, 2] | datasets: 30
Lock íntegro.


## 2. Executar a avaliação final

Antes, confirme que o lock está comitado (`git log -- results/protocolo.lock`). A célula abaixo executa `final_test` (equivalente a `python -m src.pipeline.final_test` na raiz). Ela pode levar horas; prefira executá-la no terminal. **Execute uma única vez** e, em seguida, comite `results/`.

In [3]:
EXECUTAR = False   # mude para True conscientemente, uma única vez

if EXECUTAR:
    final_test.main(['--output-dir', str(RESULTS)], root=REPO)   # recusa se houver vestígio de avaliação
else:
    print('Avaliação final não executada nesta célula.')

Avaliação final não executada nesta célula.


## 3. Ler os resultados, o log e o manifesto

In [4]:
test = pd.read_csv(RESULTS / 'test.csv')
print(test['status'].value_counts())
test.groupby('model')[['auc_ovo', 'accuracy', 'g_mean', 'cross_entropy', 'total_time_s']].mean()

status
ok    450
Name: count, dtype: int64


,auc_ovo,accuracy,g_mean,cross_entropy,total_time_s
model,,,,,
catboost,0.840498,0.858789,0.637027,0.455063,4.732594
group_default,0.860690,0.870270,0.654843,0.300545,22.240215
group_tuned,0.860621,0.870410,0.654129,0.314971,218.781091
lightgbm,0.840691,0.857008,0.618673,0.344724,3.319299
xgboost,0.828100,0.854774,0.616573,0.427733,2.958593


In [5]:
display(pd.read_csv(RESULTS / 'test_log.csv'))
manifest = json.loads((RESULTS / 'test_manifest.json').read_text(encoding='utf-8'))
states = pd.Series(manifest['status']).value_counts().to_dict()
print('lock usado:', manifest['lock_sha256'], '| execuções:', len(manifest['runs']),
      '| commits (git_head):', [r['git_head'][:10] for r in manifest['runs']], '| estados:', states)

,timestamp,event,lock_sha256,git_head,model,status,detail
0,2026-11-07T18:16:14+00:00,inicio,59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b...,9718c9649b7c54730252336305f3ce2a8d795dec,NaN,NaN,combinacoes=450; resume=False
1,2026-11-07T18:16:42+00:00,sistema,59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b...,9718c9649b7c54730252336305f3ce2a8d795dec,363621|0|lightgbm,error,ValueError: Key backend: 'module://matplotlib_...
2,2026-11-07T18:16:42+00:00,sistema,59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b...,9718c9649b7c54730252336305f3ce2a8d795dec,363621|0|xgboost,error,ValueError: Key backend: 'module://matplotlib_...
3,2026-11-07T18:16:42+00:00,sistema,59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b...,9718c9649b7c54730252336305f3ce2a8d795dec,363621|0|catboost,error,ValueError: Key backend: 'module://matplotlib_...
4,2026-11-07T18:16:48+00:00,sistema,59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b...,9718c9649b7c54730252336305f3ce2a8d795dec,363621|0|group_default,ok,NaN
...,...,...,...,...,...,...,...
485,2026-11-08T01:54:18+00:00,fim,59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b...,9718c9649b7c54730252336305f3ce2a8d795dec,NaN,NaN,0 de 2 combinações ok
486,2026-11-08T02:38:27+00:00,inicio,59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b...,9718c9649b7c54730252336305f3ce2a8d795dec,NaN,NaN,combinacoes=2; resume=True
487,2026-11-08T02:49:54+00:00,sistema,59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b...,9718c9649b7c54730252336305f3ce2a8d795dec,363628|1|group_tuned,ok,NaN
488,2026-11-08T02:58:08+00:00,sistema,59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b...,9718c9649b7c54730252336305f3ce2a8d795dec,363628|2|group_tuned,ok,NaN


lock usado: 59901430bd8c648642b9edff4d5ef667bd0c31ff24b11b8bb3a717b06fc69532 | execuções: 5 | commits (git_head): ['9718c9649b', '9718c9649b', '9718c9649b', '9718c9649b', '9718c9649b'] | estados: {'ok': 450}


## 4. Conferir o protocolo pelo histórico git

Depois de comitar `results/`, `check_protocol` deve terminar com **APROVADO**. A mesma verificação é feita na correção.

In [6]:
print(check_protocol.check_protocol(RESULTS, repo=REPO).text())

[OK] Primeiro commit com a avaliação final: 2d81dba2e0.
[OK] protocolo.lock comitado antes do teste: 84e60f6e93.
[OK] Hashes de 30 arquivos travados conferem no commit do lock.
[OK] Hashes dos arquivos travados conferem em HEAD (exceto alterações justificadas).
[OK] test_log.csv registra uma única avaliação inicial e 4 retomada(s).
[OK] As retomadas avaliaram apenas combinações pendentes ou com erro.
[FALHA] Há execuções da avaliação final com um protocolo.lock diferente do comitado.
[FALHA] Execuções da avaliação final em commits sem o lock comitado: ['9718c9649b7c54730252336305f3ce2a8d795dec'].
[FALHA] test_manifest.json foi gerado com outro protocolo.lock.
[OK] test.csv confere com os hashes por linha do manifesto.
Resultado: REPROVADO (3 falha(s))


A análise estatística (Demšar, bootstrap e regimes) está no notebook 04, que lê `results/test.csv`. Compare, de forma descritiva, a AUC média do teste com a da validação interna (`results/validation.csv`): diferenças grandes indicam otimismo na fase de desenvolvimento e devem ser discutidas no relatório.